# Practical 5: Feature Engineering & Unsupervised Anomaly Detection
**Course:** Python for Data Science (`BE05000231_PDS`)  
**Semester:** 5th Semester B.E. / B.Tech  
**Topic:** Canonical Feature Extraction, Shannon Entropy, Temporal Rolling Aggregations, and Isolation Forest Outlier Detection  
**Pipeline Input:** Practical 4 Labeled Dataset (`labeled_logs.csv` — 300,010 records)  
**External Real-World Benchmark:** Elastic Official Production Web Log (`elastic_labeled_logs.csv` — 10,000 records)  

---

## 1. AIM & OBJECTIVES
### Aim
To extract mathematically grounded, non-leaky statistical, lexical, temporal, and behavioral features from preprocessed web access logs, and execute unsupervised anomaly detection using Isolation Forest to identify outliers and malicious patterns.

### Objectives
1. **Pipeline Continuity:** Ingest the ground-truth labeled log records from Practical 4 (`labeled_logs.csv`).
2. **30 Canonical Feature Extraction:** Extract non-circular features across 4 orthogonal dimensions:
   - **9 Lexical & Path Features:** URL length, path depth, query parameter count, character distribution ratios, double slash counts, percent encoding density.
   - **7 Payload Complexity & Semantics:** Payload length, Shannon character entropy, SQL/Script/Traversal/Command semantic markers.
   - **7 Temporal & Session Dynamics:** Inter-request delta ($\Delta t$), rolling 1-min & 5-min request velocities, burst ratios, unique path counts, failed login attempts.
   - **7 Header & Network Telemetry:** Status code, error flags, bytes sent, user-agent length, user-agent entropy, scanner/bot indicators.
3. **Unsupervised Outlier Isolation:** Fit a scikit-learn `IsolationForest` to assign continuous `anomaly_score` values and binary `is_anomaly` flags without seeing labels during training.
4. **Outlier Attribution Analysis:** Evaluate which features correlate most strongly with detected anomalies.
5. **Dual-Format Serialization:** Export the feature-engineered dataset to CSV and Snappy-compressed Apache Parquet.
6. **External Production Benchmark:** Validate the feature pipeline and anomaly detector against Elastic production telemetry.

---
## 2. THEORETICAL FOUNDATION: FEATURE ENGINEERING IN CYBERSECURITY

### 2.1 The Shannon Entropy Metric
Character-level Shannon Entropy measures the uncertainty and randomness in a string:

$$H(X) = - \sum_{i=1}^{n} P(x_i) \log_2 P(x_i)$$

- **Normal Web Paths:** Typically show entropy between $2.8$ and $3.8$ bits/char (structured English words and dictionary paths).
- **Obfuscated Payloads (SQLi, WebShells, XSS):** Exhibit entropy $> 4.3$ bits/char due to URL-encoded hex characters (`%27%3B%20`), nested brackets, and high special-character ratios.

### 2.2 Canonical 30-Feature Inventory
| Feature Group | Count | Key Features Included |
|---|---|---|
| **1. Lexical & Path** | 9 | `url_length`, `url_entropy`, `path_depth`, `query_param_count`, `special_char_ratio`, `digit_ratio`, `uppercase_ratio`, `double_slash_count`, `percent_encoding_count` |
| **2. Payload Semantics** | 7 | `payload_length`, `payload_entropy`, `has_sql_semantics`, `has_script_semantics`, `has_traversal_semantics`, `has_command_semantics`, `has_internal_ip_semantics` |
| **3. Temporal & Session** | 7 | `inter_request_time`, `requests_per_ip_1min`, `requests_per_ip_5min`, `request_velocity_1sec`, `burst_ratio`, `unique_resources_per_ip_5min`, `failed_login_count` |
| **4. Network & Headers** | 7 | `status_code`, `is_error_status`, `bytes_sent`, `user_agent_length`, `user_agent_entropy`, `is_known_scanner`, `is_bot` |

---
## 3. IMPORT LIBRARIES & SETUP ENVIRONMENT
We import `pandas`, `numpy`, `scikit-learn`, and our canonical feature extraction module.

In [ ]:
import sys
import os
import time
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

# Resolve directories
PRACTICAL_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
WORKSPACE_DIR = PRACTICAL_DIR.parent

sys.path.insert(0, str(WORKSPACE_DIR))
from src.features.feature_definitions import (
    CANONICAL_FEATURE_NAMES,
    extract_lexical_features,
    extract_payload_semantics,
    extract_header_features
)

PROCESSED_P4 = WORKSPACE_DIR / 'Practical 4' / 'data' / 'processed'
PROCESSED_P5 = PRACTICAL_DIR / 'data' / 'processed'
PLOTS_DIR = PRACTICAL_DIR / 'outputs' / 'plots'
REPORTS_DIR = PRACTICAL_DIR / 'outputs' / 'reports'

PROCESSED_P5.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

PRIMARY_INPUT = PROCESSED_P4 / 'labeled_logs.csv'
EXTERNAL_INPUT = PROCESSED_P4 / 'elastic_labeled_logs.csv'

print(f"Practical 5 Initialized! Ingesting: {PRIMARY_INPUT.name} (Exists: {PRIMARY_INPUT.exists()})")


---
## 4. INGEST PRACTICAL 4 DATASET & EXTRACT 30 CANONICAL FEATURES
We load the labeled DataFrame and compute all 30 features across lexical, payload, temporal, and header dimensions.

In [ ]:
t0 = time.time()
df = pd.read_csv(PRIMARY_INPUT)
print(f"Loaded Primary Dataset: {len(df):,} rows x {len(df.columns)} columns in {time.time()-t0:.2f}s")

# 1. Lexical Features (Cached)
lex_cache = {}
lex_recs = []
for u in df['resource_requested']:
    if u not in lex_cache:
        lex_cache[u] = extract_lexical_features(u)
    lex_recs.append(lex_cache[u])
df_lex = pd.DataFrame(lex_recs, index=df.index)

# 2. Payload Features (Cached)
payload_cache = {}
payload_recs = []
for p in df['payload'].fillna(''):
    p_str = str(p)
    if p_str not in payload_cache:
        payload_cache[p_str] = extract_payload_semantics(p_str)
    payload_recs.append(payload_cache[p_str])
df_payload = pd.DataFrame(payload_recs, index=df.index)

# 3. Header Features
header_recs = []
for s, b, ua in zip(df['status_code'].fillna(200).astype(int), df['bytes_sent'].fillna(0).astype(int), df['user_agent'].fillna('unknown').astype(str)):
    header_recs.append(extract_header_features(s, b, ua))
df_header = pd.DataFrame(header_recs, index=df.index)

# 4. Temporal Features
df['_dt'] = pd.to_datetime(df['datetime_utc'], errors='coerce')
df_sorted = df.sort_values(['client_ip', '_dt']).copy()
df_sorted['inter_request_time'] = df_sorted.groupby('client_ip')['_dt'].diff().dt.total_seconds().fillna(60.0).clip(lower=0.0)
df_sorted['is_failed_auth'] = df_sorted['status_code'].isin([401, 403]).astype(np.int8)

df_indexed = df_sorted.set_index('_dt')
grp_ip = df_indexed.groupby('client_ip')

df_sorted['requests_per_ip_1min'] = grp_ip['line_number'].rolling('1min').count().values.astype(np.int32)
df_sorted['requests_per_ip_5min'] = grp_ip['line_number'].rolling('5min').count().values.astype(np.int32)
df_sorted['failed_login_count'] = grp_ip['is_failed_auth'].rolling('5min').sum().values.astype(np.int32)
df_sorted['request_velocity_1sec'] = (df_sorted['inter_request_time'] <= 1.0).astype(np.int8)
avg_5m = (df_sorted['requests_per_ip_5min'] / 5.0).clip(lower=1.0)
df_sorted['burst_ratio'] = (df_sorted['requests_per_ip_1min'] / avg_5m).round(3)

df_sorted['path_id'] = pd.factorize(df_sorted['clean_path'])[0].astype(np.float64)
df_sorted['unique_resources_per_ip_5min'] = grp_ip['path_id'].rolling('5min').apply(lambda s: len(np.unique(s)), raw=True).values.astype(np.int32)

df_temporal = df_sorted.loc[df.index, [
    'inter_request_time', 'requests_per_ip_1min', 'requests_per_ip_5min',
    'request_velocity_1sec', 'burst_ratio', 'unique_resources_per_ip_5min', 'failed_login_count'
]]

df.drop(columns=['_dt'], inplace=True, errors='ignore')
features_df = pd.concat([df_lex, df_payload, df_temporal, df_header], axis=1)[CANONICAL_FEATURE_NAMES]
df_featured = pd.concat([df, features_df], axis=1)

print(f"Extracted {len(CANONICAL_FEATURE_NAMES)} canonical features in {time.time()-t0:.2f}s!")
df_featured[CANONICAL_FEATURE_NAMES].head(3)


---
## 5. UNSUPERVISED ANOMALY DETECTION (ISOLATION FOREST)
We train an unsupervised Isolation Forest on the 30 canonical features to detect outliers and anomalous behaviors.

In [ ]:
t_iso = time.time()
X = df_featured[CANONICAL_FEATURE_NAMES].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(n_estimators=100, max_samples=256, contamination=0.10, random_state=42, n_jobs=-1)
iso.fit(X_scaled)

df_featured['anomaly_score'] = np.round(-iso.score_samples(X_scaled), 4)
df_featured['is_anomaly'] = (iso.predict(X_scaled) == -1).astype(np.int8)

outliers = int(df_featured['is_anomaly'].sum())
pct = (outliers / len(df_featured)) * 100
auc = roc_auc_score(df_featured['is_attack'], df_featured['anomaly_score'])

print(f"Isolation Forest Completed in {time.time()-t_iso:.2f}s!")
print(f"Flagged Outliers    : {outliers:,} ({pct:.2f}% contamination)")
print(f"ROC-AUC vs Attacks  : {auc:.4f} (Unsupervised alignment with malicious traffic)")
print(f"Mean Anomaly Score  : {df_featured['anomaly_score'].mean():.4f}")


---
## 6. EXTERNAL BENCHMARK: ELASTIC PRODUCTION TELEMETRY
We extract features and execute the Isolation Forest anomaly detector on Elastic's official production dataset.

In [ ]:
df_ext = pd.read_csv(EXTERNAL_INPUT)
print(f"External Elastic Dataset: {len(df_ext):,} records")

# Lexical
df_ext_lex = pd.DataFrame([extract_lexical_features(u) for u in df_ext['resource_requested']], index=df_ext.index)
# Payload
df_ext_payload = pd.DataFrame([extract_payload_semantics(str(p)) for p in df_ext['payload'].fillna('')], index=df_ext.index)
# Header
df_ext_header = pd.DataFrame([extract_header_features(s, b, ua) for s, b, ua in zip(df_ext['status_code'].fillna(200).astype(int), df_ext['bytes_sent'].fillna(0).astype(int), df_ext['user_agent'].fillna('unknown').astype(str))], index=df_ext.index)

# Temporal
df_ext['_dt'] = pd.to_datetime(df_ext['datetime_utc'], errors='coerce')
df_ext_s = df_ext.sort_values(['client_ip', '_dt']).copy()
df_ext_s['inter_request_time'] = df_ext_s.groupby('client_ip')['_dt'].diff().dt.total_seconds().fillna(60.0).clip(lower=0.0)
df_ext_s['is_failed_auth'] = df_ext_s['status_code'].isin([401, 403]).astype(np.int8)
df_ext_idx = df_ext_s.set_index('_dt')
grp_ext = df_ext_idx.groupby('client_ip')
df_ext_s['requests_per_ip_1min'] = grp_ext['line_number'].rolling('1min').count().values.astype(np.int32)
df_ext_s['requests_per_ip_5min'] = grp_ext['line_number'].rolling('5min').count().values.astype(np.int32)
df_ext_s['failed_login_count'] = grp_ext['is_failed_auth'].rolling('5min').sum().values.astype(np.int32)
df_ext_s['request_velocity_1sec'] = (df_ext_s['inter_request_time'] <= 1.0).astype(np.int8)
avg_ext_5m = (df_ext_s['requests_per_ip_5min'] / 5.0).clip(lower=1.0)
df_ext_s['burst_ratio'] = (df_ext_s['requests_per_ip_1min'] / avg_ext_5m).round(3)
df_ext_s['path_id'] = pd.factorize(df_ext_s['clean_path'])[0].astype(np.float64)
df_ext_s['unique_resources_per_ip_5min'] = grp_ext['path_id'].rolling('5min').apply(lambda s: len(np.unique(s)), raw=True).values.astype(np.int32)

df_ext_temporal = df_ext_s.loc[df_ext.index, ['inter_request_time', 'requests_per_ip_1min', 'requests_per_ip_5min', 'request_velocity_1sec', 'burst_ratio', 'unique_resources_per_ip_5min', 'failed_login_count']]
df_ext.drop(columns=['_dt'], inplace=True, errors='ignore')

ext_features = pd.concat([df_ext_lex, df_ext_payload, df_ext_temporal, df_ext_header], axis=1)[CANONICAL_FEATURE_NAMES]
df_ext_featured = pd.concat([df_ext, ext_features], axis=1)

X_ext_scaled = scaler.transform(df_ext_featured[CANONICAL_FEATURE_NAMES].fillna(0).values)
df_ext_featured['anomaly_score'] = np.round(-iso.score_samples(X_ext_scaled), 4)
df_ext_featured['is_anomaly'] = (iso.predict(X_ext_scaled) == -1).astype(np.int8)

print(f"External Outliers Flagged: {int(df_ext_featured['is_anomaly'].sum()):,} ({df_ext_featured['is_anomaly'].mean()*100:.2f}%)")


---
## 7. EXPORT FEATURED DATASETS (CSV & PARQUET)
We export the feature-engineered datasets ready for Practical 6.

In [ ]:
csv_out = PROCESSED_P5 / 'featured_logs.csv'
pq_out = PROCESSED_P5 / 'featured_logs.parquet'
ext_out = PROCESSED_P5 / 'elastic_featured_logs.csv'

df_featured.to_csv(csv_out, index=False)
df_featured.to_parquet(pq_out, index=False, compression='snappy')
df_ext_featured.to_csv(ext_out, index=False)

print(f"Exported Primary CSV     : {csv_out.name} ({csv_out.stat().st_size / (1024*1024):.2f} MB)")
print(f"Exported Primary Parquet : {pq_out.name} ({pq_out.stat().st_size / (1024*1024):.2f} MB)")
print(f"Exported External CSV    : {ext_out.name} ({ext_out.stat().st_size / (1024*1024):.2f} MB)")


---
## 8. PUBLICATION-QUALITY DIAGNOSTIC VISUALIZATIONS
We generate high-resolution exploratory diagnostic plots for feature correlations, entropy, and PCA projections.

In [ ]:
# 1. Shannon Entropy Boxplot
plt.figure(figsize=(10, 4.5), dpi=150)
top_cls = df_featured['label'].value_counts().head(7).index
entropy_vals = [df_featured[df_featured['label'] == c]['url_entropy'].values for c in top_cls]
plt.boxplot(entropy_vals, tick_labels=top_cls, patch_artist=True, boxprops=dict(facecolor='#1687FF', color='#060912'), medianprops=dict(color='#FF5577', linewidth=2))
plt.title('Shannon Entropy Distribution Across Top Classes', fontsize=12, fontweight='bold')
plt.ylabel('Entropy (Bits / Character)')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# 2. Anomaly Score Density Distribution
plt.figure(figsize=(9, 4), dpi=150)
plt.hist(df_featured[df_featured['is_attack'] == 0]['anomaly_score'], bins=50, alpha=0.6, label='Benign Traffic', color='#10B981', density=True)
plt.hist(df_featured[df_featured['is_attack'] == 1]['anomaly_score'], bins=50, alpha=0.6, label='Malicious Attacks', color='#EF4444', density=True)
plt.title('Isolation Forest Anomaly Score Distribution (Density)', fontsize=12, fontweight='bold')
plt.xlabel('Anomaly Score')
plt.ylabel('Probability Density')
plt.legend()
plt.tight_layout()
plt.show()
